# Potential Talents Robust Demo

This notebook demonstrates an end-to-end robust ranking workflow:

1. Robust ingestion and validation
2. Modern embedding-based ranking with calibrated `fit` score (0-1)
3. Feedback-driven reranking (star/skip)
4. Ranking metrics (NDCG@K, MAP@K)
5. Cutoff and out-of-scope filtering

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

from talentfit.ingest import load_candidates
from talentfit.ranker import PotentialTalentsRanker
from talentfit.cutoff import CutoffPolicy
from talentfit.eval import ndcg_at_k, map_at_k
from talentfit.reporting import simple_impact_report

pd.set_option('display.max_colwidth', 120)
PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
print('Project root:', PROJECT_ROOT)

## 1) Data Ingestion and Validation

In [ ]:
data_path = PROJECT_ROOT / 'potential-talents.xlsx'
res = load_candidates(data_path)
df = res.df

print('Shape:', df.shape)
print('Columns:', list(df.columns))
print('\nMissing values:\n', df.isna().sum())
print('\nConnection parsing preview:')
df[['id', 'job_title', 'connection_raw', 'connections']].head(10)

## 2) Rank Candidates for a Query

In [ ]:
queries = ['aspiring human resources', 'seeking human resources']
ranker = PotentialTalentsRanker()

ranked_before = ranker.rank(queries, top_k=50)
cutoff = CutoffPolicy().decide(ranked_before['fit'].to_numpy())
ranked_before['keep'] = (ranked_before['fit'] >= cutoff).astype(int)

print(f'Cutoff fit score: {cutoff:.4f}')
print(simple_impact_report(ranked_before))
ranked_before[['id', 'job_title', 'fit', 'embed_similarity', 'is_out_of_scope', 'keep']].head(15)

## 3) Simulate Reviewer Feedback (Star/Skip) and Re-rank

In [ ]:
# Example feedback: star one ideal candidate, skip a couple less-ideal ones.
# Replace these IDs after manual review in your real workflow.
starred_ids = [99]
skipped_ids = [6, 49]

ranker.record_feedback(queries, starred_ids=starred_ids, skipped_ids=skipped_ids)
ranked_after = ranker.rank(queries, top_k=50)

ranked_after[['id', 'job_title', 'fit', 'embed_similarity']].head(15)

## 4) Evaluate Ranking Improvement (NDCG@K / MAP@K)

Define one or more "ideal" candidates based on reviewer judgment, then compare metrics before and after feedback.

In [ ]:
k = 10
ideal_ids = {99}

ids_before = ranked_before['id'].tolist()
ids_after = ranked_after['id'].tolist()

rel_before = np.array([1 if i in ideal_ids else 0 for i in ids_before], dtype=int)
rel_after = np.array([1 if i in ideal_ids else 0 for i in ids_after], dtype=int)

results = pd.DataFrame([
    {'stage': 'before_feedback', 'ndcg_at_k': ndcg_at_k(rel_before, k), 'map_at_k': map_at_k(rel_before, k)},
    {'stage': 'after_feedback', 'ndcg_at_k': ndcg_at_k(rel_after, k), 'map_at_k': map_at_k(rel_after, k)},
])
results

## 5) Notes for Production Use

- Keep collecting star/skip feedback per role query.
- Retrain/calibrate periodically as feedback volume grows.
- Keep location out of scoring by default to reduce bias risk.
- Track cutoff acceptance rates and manual override rates by query type.